## **Pulizia dati Progetto Finale**

In [42]:
# Importazione Librerie
import pandas as pd
import numpy as np

In [43]:
dist_center_raw=pd.read_csv("..\\data\\initial_data\\Distribution Center.csv")
events_raw=pd.read_csv("..\\data\\initial_data\\events.csv")
inv_items_raw=pd.read_csv("..\\data\\initial_data\\inventory_items.csv")
order_items_raw=pd.read_csv("..\\data\\initial_data\\order_items.csv")
orders_raw=pd.read_csv("..\\data\\initial_data\\orders.csv")
prod_raw=pd.read_csv("..\\data\\initial_data\\products.csv")
users_raw=pd.read_csv("..\\data\\initial_data\\Users.csv")

In [44]:
dist_center=dist_center_raw.copy()
events=events_raw.copy()
inv_items=inv_items_raw.copy()
order_items=order_items_raw.copy()
orders=orders_raw.copy()
prod=prod_raw.copy()
users=users_raw.copy()

### Distribution Center

In [45]:
dist_center.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      10 non-null     int64
 1   name    10 non-null     str  
dtypes: int64(1), str(1)
memory usage: 292.0 bytes


In [46]:
dist_center

,id,name
0,3,Houston TX
1,7,Philadelphia PA
2,5,New Orleans LA
3,4,Los Angeles CA
4,8,Mobile AL
5,1,Memphis TN
6,2,Chicago IL
7,6,Port Authority of New York/New Jersey NY/NJ
8,10,Savannah GA
9,9,Charleston SC


In [47]:
# Nessuna pulizia necessaria

In [48]:
dist_center.to_csv("..\\data\\clean_data\\distribution_center_clean.csv", index=False)

### Events

In [49]:
events.info()
# Dati Mancanti in user_id (utenti non registrati)

<class 'pandas.DataFrame'>
RangeIndex: 681828 entries, 0 to 681827
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   session_id      681828 non-null  str    
 1   user_id         181828 non-null  float64
 2   finish          681828 non-null  int64  
 3   event_type      681828 non-null  str    
 4   traffic_source  681828 non-null  str    
 5   state           681828 non-null  str    
dtypes: float64(1), int64(1), str(4)
memory usage: 31.2 MB


In [50]:
events.sample(5)

,session_id,user_id,finish,event_type,traffic_source,state
675667,ae46dbdc-66e4-4608-8db5-499764764d7f,15405.0,13,purchase,Adwords,Guangdong
273480,b8bc29ba-c819-48db-b9d9-61297d5227c2,NaN,3,cancel,Email,Guangxi Zhuang Autonomous Region
276544,0c8aa4c8-ecd6-4571-8305-11d941f4a4e2,NaN,3,cancel,Adwords,Rio Grande do Sul
499544,a219344e-c169-42a5-9681-ce2d7f48cb2e,NaN,3,cart,Email,Washington
606384,b13b7867-e8bb-4543-a3fa-a04b87484ee7,94162.0,7,purchase,Email,Espírito Santo


In [51]:
events["user_id"] = events["user_id"].fillna("Unknown")
events["user_id"] = events["user_id"].astype("string")

In [52]:
# Aggiungiamo il country sfruttanfo state, che si trova anche nella tabella users
city_country_map = users[['state', 'country']].drop_duplicates()
events = events.merge(city_country_map, on='state', how='left')

In [100]:
events.sample(5)

,session_id,user_id,finish,event_type,traffic_source,state,country
344582,5773c49b-20eb-4673-bae6-0010194b3e0b,Unknown,3,cart,Adwords,Busan,South Korea
537822,2caf1543-c060-4c88-8cf1-eb5a3269a5fa,42804.0,5,purchase,Facebook,California,United States
278101,ff6bb752-f287-45a7-8ed6-7877125486c6,Unknown,3,cancel,Organic,Fujian,China
577095,15245d2a-49af-4ff3-a393-9c141efde39b,69452.0,5,purchase,Organic,Liaoning,China
351560,eb09fd4b-7107-434b-aa3b-11f8d8c7be12,Unknown,3,cancel,Email,Incheon Metropolitan City,South Korea


In [54]:
events.info()
# qualche dato mancante sul country

<class 'pandas.DataFrame'>
RangeIndex: 698847 entries, 0 to 698846
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   session_id      698847 non-null  str   
 1   user_id         698847 non-null  string
 2   finish          698847 non-null  int64 
 3   event_type      698847 non-null  str   
 4   traffic_source  698847 non-null  str   
 5   state           698847 non-null  str   
 6   country         698823 non-null  str   
dtypes: int64(1), str(5), string(1)
memory usage: 37.3 MB


In [55]:
events["country"]=events["country"].fillna("Unknown")

In [56]:
events.to_csv("..\\data\\clean_data\\events_clean.csv", index=False)

### Inventory Items

In [57]:
inv_items.info()
# I Dati Mancanti in sold_at vanno mantenuti così come sono, il dato mancante ci fa capire che un pezzo è ancora in stock
# Cambiare created_at e sold_at in datetime

<class 'pandas.DataFrame'>
RangeIndex: 489823 entries, 0 to 489822
Data columns (total 5 columns):
 #   Column                          Non-Null Count   Dtype
---  ------                          --------------   -----
 0   id                              489823 non-null  int64
 1   product_id                      489823 non-null  int64
 2   created_at                      489823 non-null  str  
 3   sold_at                         181404 non-null  str  
 4   product_distribution_center_id  489823 non-null  int64
dtypes: int64(3), str(2)
memory usage: 18.7 MB


In [58]:
inv_items["created_at"] = inv_items["created_at"].str.replace(r"\.\d+", "", regex=True)
inv_items["created_at"] = pd.to_datetime(inv_items["created_at"], utc=True)
inv_items["sold_at"] = inv_items["sold_at"].str.replace(r"\.\d+", "", regex=True)
inv_items["sold_at"] = pd.to_datetime(inv_items["sold_at"], utc=True)

In [59]:
inv_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 489823 entries, 0 to 489822
Data columns (total 5 columns):
 #   Column                          Non-Null Count   Dtype              
---  ------                          --------------   -----              
 0   id                              489823 non-null  int64              
 1   product_id                      489823 non-null  int64              
 2   created_at                      489823 non-null  datetime64[us, UTC]
 3   sold_at                         181404 non-null  datetime64[us, UTC]
 4   product_distribution_center_id  489823 non-null  int64              
dtypes: datetime64[us, UTC](2), int64(3)
memory usage: 18.7 MB


In [60]:
inv_items.sample(5)

,id,product_id,created_at,sold_at,product_distribution_center_id
94648,280430,2303,2022-03-24 16:25:00+00:00,NaT,7
150705,66749,20624,2020-04-29 16:43:00+00:00,NaT,4
196829,192961,15224,2025-10-01 19:42:43+00:00,2025-11-25 04:05:43+00:00,3
349021,236420,25264,2025-01-03 12:47:00+00:00,NaT,1
100360,412436,1937,2025-11-14 18:17:43+00:00,2025-12-01 19:25:43+00:00,1


In [61]:
# Se creo una variabile che assume valore 0 se il prodotto è stato venduto e 1 se è ancora in stock, posso eliminare sold_at
# Inoltre se aggiungo il mese e l'anno di creazione posso eliminare anche created_at
inv_items['in_stock'] = inv_items['sold_at'].isna().astype(int)
inv_items['month_year_creation'] = inv_items['created_at'].dt.strftime('%b %Y')
# magari posso aggiungere la durata di permanenza in stock
inv_items["time_in_stock"] = inv_items["sold_at"] - inv_items["created_at"]
inv_items["days_in_stock"] = inv_items["time_in_stock"].dt.days

In [62]:
inv_items=inv_items[["id", "product_id", "product_distribution_center_id", "in_stock", "month_year_creation", "days_in_stock"]]
# days_in_stock avrà valore null se non è ancora stato venduto

In [63]:
inv_items.sample(5)

,id,product_id,product_distribution_center_id,in_stock,month_year_creation,days_in_stock
405696,357254,1498,1,0,Aug 2023,30.0
121742,181422,11096,2,1,Jun 2023,NaN
379004,219884,802,2,1,Dec 2025,NaN
358036,275096,9190,5,0,Dec 2025,20.0
367558,133469,20270,9,1,May 2024,NaN


In [64]:
inv_items.to_csv("..\\data\\clean_data\\inventory_items_clean.csv", index=False)

### Order Items

In [65]:
order_items.info()
# Dati Mancanti solo in shipped_at, delivered_at e returned_at. Vanno mantenuti, servono per capire lo stato dei vari prodotti dell'ordine
# Cambiare le variabili con date in DateTime

<class 'pandas.DataFrame'>
RangeIndex: 181404 entries, 0 to 181403
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 181404 non-null  int64  
 1   order_id           181404 non-null  int64  
 2   user_id            181404 non-null  int64  
 3   product_id         181404 non-null  int64  
 4   inventory_item_id  181404 non-null  int64  
 5   status             181404 non-null  str    
 6   created_at         181404 non-null  str    
 7   shipped_at         118292 non-null  str    
 8   delivered_at       63933 non-null   str    
 9   returned_at        18372 non-null   str    
 10  sale_price         181404 non-null  float64
dtypes: float64(1), int64(5), str(5)
memory usage: 15.2 MB


In [66]:
order_items["created_at"] = order_items["created_at"].str.replace(r"\.\d+", "", regex=True)
order_items["created_at"] = pd.to_datetime(order_items["created_at"], utc=True)
order_items["shipped_at"] = order_items["shipped_at"].str.replace(r"\.\d+", "", regex=True)
order_items["shipped_at"] = pd.to_datetime(order_items["shipped_at"], utc=True)
order_items["delivered_at"] = order_items["delivered_at"].str.replace(r"\.\d+", "", regex=True)
order_items["delivered_at"] = pd.to_datetime(order_items["delivered_at"], utc=True)
order_items["returned_at"] = order_items["returned_at"].str.replace(r"\.\d+", "", regex=True)
order_items["returned_at"] = pd.to_datetime(order_items["returned_at"], utc=True)

In [67]:
# Arrotondiamo sale_price
order_items["sale_price"]=order_items["sale_price"].round(2)

In [68]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 181404 entries, 0 to 181403
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   id                 181404 non-null  int64              
 1   order_id           181404 non-null  int64              
 2   user_id            181404 non-null  int64              
 3   product_id         181404 non-null  int64              
 4   inventory_item_id  181404 non-null  int64              
 5   status             181404 non-null  str                
 6   created_at         181404 non-null  datetime64[us, UTC]
 7   shipped_at         118292 non-null  datetime64[us, UTC]
 8   delivered_at       63933 non-null   datetime64[us, UTC]
 9   returned_at        18372 non-null   datetime64[us, UTC]
 10  sale_price         181404 non-null  float64            
dtypes: datetime64[us, UTC](4), float64(1), int64(5), str(1)
memory usage: 15.2 MB


In [69]:
order_items.sample(5)

,id,order_id,user_id,product_id,inventory_item_id,status,created_at,shipped_at,delivered_at,returned_at,sale_price
159040,170075,117246,93841,3060,459198,Shipped,2024-12-21 02:36:26+00:00,2024-12-23 12:57:20+00:00,NaT,NaT,110.00
75502,165387,113966,91188,27444,446540,Complete,2025-06-05 05:01:50+00:00,2025-06-01 18:36:22+00:00,2025-06-04 01:11:22+00:00,NaT,34.16
128921,56275,38993,31371,21067,151773,Complete,2025-09-27 16:31:29+00:00,2025-09-29 20:15:40+00:00,2025-10-01 09:55:40+00:00,NaT,64.00
3541,51223,35582,28669,15713,138113,Shipped,2025-07-19 17:31:05+00:00,2025-07-16 15:27:10+00:00,NaT,NaT,7.99
126231,165508,114050,91244,28242,446873,Complete,2022-07-13 18:46:15+00:00,2022-07-14 12:43:37+00:00,2022-07-15 23:43:37+00:00,NaT,60.00


In [70]:
# Ci servono created_at, shipped_at, delivered_at, returned_at? Qui stiamo parlando dei singoli prodotti degli ordini. 
# A noi interessano le tempistiche degli ordini, non dei singoli prodotti. Le togliamo?

In [71]:
# order_items=order_items[["id", "order_id", "user_id","product_id","inventory_item_id","status", "created_at","shipped_at","delivered_at", "returned_at","sale_price"]]

In [72]:
# order_items.sample(5)

In [73]:
order_items.to_csv("..\\data\\clean_data\\order_items_clean.csv", index=False)

### Orders

In [74]:
orders.info()
# Dati Mancanti solo in shipped_at, delivered_at e returned_at. Vanno mantenuti, servono per capire lo stato dell'ordine
# Cambiare in datetime le variabili con le date

<class 'pandas.DataFrame'>
RangeIndex: 124988 entries, 0 to 124987
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   order_id      124988 non-null  int64
 1   user_id       124988 non-null  int64
 2   status        124988 non-null  str  
 3   created_at    124988 non-null  str  
 4   returned_at   12635 non-null   str  
 5   shipped_at    81363 non-null   str  
 6   delivered_at  43863 non-null   str  
 7   num_of_item   124988 non-null  int64
dtypes: int64(3), str(5)
memory usage: 7.6 MB


In [75]:
orders["created_at"] = orders["created_at"].str.replace(r"\.\d+", "", regex=True)
orders["created_at"] = pd.to_datetime(orders["created_at"], utc=True)
orders["shipped_at"] = orders["shipped_at"].str.replace(r"\.\d+", "", regex=True)
orders["shipped_at"] = pd.to_datetime(orders["shipped_at"], utc=True)
orders["returned_at"] = orders["returned_at"].str.replace(r"\.\d+", "", regex=True)
orders["returned_at"] = pd.to_datetime(orders["returned_at"], utc=True)
orders["delivered_at"] = orders["delivered_at"].str.replace(r"\.\d+", "", regex=True)
orders["delivered_at"] = pd.to_datetime(orders["delivered_at"], utc=True)

In [76]:
orders.sample(5)

,order_id,user_id,status,created_at,returned_at,shipped_at,delivered_at,num_of_item
101250,25173,20269,Returned,2019-05-20 11:11:22+00:00,2019-05-25 23:16:22+00:00,2019-05-21 20:06:22+00:00,2019-05-25 02:22:22+00:00,1
5500,72721,58199,Cancelled,2023-11-30 04:10:26+00:00,NaT,NaT,NaT,1
119455,88291,70643,Shipped,2020-08-15 11:11:57+00:00,NaT,2020-08-18 02:05:57+00:00,NaT,1
121712,103700,82980,Shipped,2022-05-31 17:39:54+00:00,NaT,2022-06-02 06:21:54+00:00,NaT,1
65616,41220,33126,Cancelled,2019-12-12 00:32:05+00:00,NaT,NaT,NaT,2


In [77]:
# Se aggiungiamo il mese e l'anno di creazione. E calcoliamo il tempo per spedire un ordine e per farlo arrivare:
# Ci servono ancora created_at, shipped_at, delivered_at, returned_at?

In [78]:
# Aggiunta mese e anno di creazione ordine
orders['month_year_creation'] = orders['created_at'].dt.strftime('%b %Y')
# Aggiunta tempo di spedizione e di consegna
orders["time_for_shipping"] = orders["shipped_at"] - orders["created_at"]
orders["time_for_delivery"] = orders["delivered_at"] - orders["shipped_at"]
orders["days_for_shipping"] = orders["time_for_shipping"].dt.days
orders["days_for_delivery"] = orders["time_for_delivery"].dt.days

In [79]:
orders=orders[["order_id","user_id","status","num_of_item","month_year_creation","days_for_shipping","days_for_delivery"]]

In [80]:
orders.sample(5)
# Ovviamente rimangono dati mancanti in days_for_shipping e days_for_delivery in base allo stato dell'ordine

,order_id,user_id,status,num_of_item,month_year_creation,days_for_shipping,days_for_delivery
30973,60920,48726,Processing,1,Apr 2025,NaN,NaN
95894,84050,67247,Processing,1,Apr 2023,NaN,NaN
50710,45385,36425,Shipped,1,Sep 2024,1.0,NaN
69096,88271,70627,Cancelled,1,Feb 2026,NaN,NaN
47675,25709,20690,Shipped,2,Mar 2023,2.0,NaN


In [81]:
"""
Vogliamo aggiungere il costo, il prezzo di vendita e il profitto per ogni ordine.
Come possiamo fare:
1. Facciamo una join tra order_items e products
2. Manteniamo nel nuovo dataframe solo order_id, product_id, sale_price e cost
3. Calcoliamo il profitto per ogni prodotto: sale_price-cost
4. Raggruppiamo per order_id e sommiamo i costi e il profitto
5. Facciamo una join tra il nuovo dataframe e orders
"""

'\nVogliamo aggiungere il costo, il prezzo di vendita e il profitto per ogni ordine.\nCome possiamo fare:\n1. Facciamo una join tra order_items e products\n2. Manteniamo nel nuovo dataframe solo order_id, product_id, sale_price e cost\n3. Calcoliamo il profitto per ogni prodotto: sale_price-cost\n4. Raggruppiamo per order_id e sommiamo i costi e il profitto\n5. Facciamo una join tra il nuovo dataframe e orders\n'

In [82]:
df_merged1 = pd.merge(left=order_items, right=prod, how='inner', left_on='product_id', right_on='id')
df_merged1=df_merged1[["order_id","product_id","sale_price","cost"]]
df_merged1["profit_per_product"]=df_merged1["sale_price"]-df_merged1["cost"]
df_merged1.sample(5)

,order_id,product_id,sale_price,cost,profit_per_product
27866,93866,19211,17.45,9.09145,8.35855
131130,442,23265,65.00,35.10000,29.90000
32890,83584,566,19.95,11.03235,8.91765
128088,102699,9527,62.36,38.10196,24.25804
138307,98036,14786,72.00,31.60800,40.39200


In [83]:
cost_prof_per_order=df_merged1.groupby("order_id", as_index=False).agg({"sale_price":sum, "cost": sum, "profit_per_product": sum})
cost_prof_per_order = cost_prof_per_order.rename(columns={"profit_per_product": "profit"})
cost_prof_per_order["profit"]=cost_prof_per_order["profit"].round(2)
cost_prof_per_order["cost"]=cost_prof_per_order["cost"].round(2)
cost_prof_per_order.head()

,order_id,sale_price,cost,profit
0,1,73.10,34.90,38.20
1,2,156.99,67.52,89.47
2,3,29.99,15.98,14.01
3,4,80.63,44.02,36.61
4,5,158.00,89.45,68.55


In [84]:
orders=pd.merge(left=orders, right=cost_prof_per_order, how='inner', on='order_id')
orders.sample(5)
# fare attenzione allo stato dell'ordine. tenere conto del profitto solo per ordini completati, spediti o in processo.
# Per ordini cancellati il profitto si azzera.
# Per gli ordini resi il profitto potrebbe diventare negativo (profitto=-costo). Decidere insieme.

,order_id,user_id,status,num_of_item,month_year_creation,days_for_shipping,days_for_delivery,sale_price,cost,profit
113126,46010,36935,Shipped,1,Feb 2023,2.0,NaN,12.00,6.98,5.02
66184,49195,39425,Cancelled,2,Aug 2025,NaN,NaN,96.00,53.78,42.22
96599,90999,72784,Processing,1,Nov 2021,NaN,NaN,54.98,25.68,29.30
66765,56915,45474,Cancelled,1,Jul 2023,NaN,NaN,295.00,105.90,189.10
69324,91197,72930,Cancelled,1,Nov 2024,NaN,NaN,17.99,7.41,10.58


In [85]:
# Modifica profitto in base allo status dell'ordine
orders.loc[orders['status'] == 'Cancelled', 'profit'] = 0
orders.loc[orders['status'] == 'Returned', 'profit'] = -orders['cost']

In [102]:
orders.sample(5)

,order_id,user_id,status,num_of_item,month_year_creation,days_for_shipping,days_for_delivery,sale_price,cost,profit
105261,106275,85009,Returned,4,Nov 2023,2.0,1.0,175.89,75.12,-75.12
44101,1613,1279,Shipped,1,Jul 2024,0.0,NaN,124.00,78.49,45.51
83594,94279,75428,Complete,3,Mar 2023,2.0,4.0,123.95,67.03,56.92
36422,114671,91773,Processing,2,Mar 2022,NaN,NaN,90.00,43.86,46.14
123950,118337,94719,Shipped,2,Jul 2025,0.0,NaN,118.99,61.88,57.11


In [87]:
orders.to_csv("..\\data\\clean_data\\orders_clean.csv", index=False)

### Products

In [88]:
prod.info()
# name e brand con Dati Mancanti

<class 'pandas.DataFrame'>
RangeIndex: 29120 entries, 0 to 29119
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      29120 non-null  int64  
 1   cost                    29120 non-null  float64
 2   category                29120 non-null  str    
 3   name                    29118 non-null  str    
 4   brand                   29096 non-null  str    
 5   retail_price            29120 non-null  float64
 6   department              29120 non-null  str    
 7   sku                     29120 non-null  str    
 8   distribution_center_id  29120 non-null  int64  
dtypes: float64(2), int64(2), str(5)
memory usage: 2.0 MB


In [89]:
prod["name"]=prod["name"].fillna("Unknown")
prod["brand"]=prod["brand"].fillna("Unknown")

In [90]:
prod.sample(5)

,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id
24923,16303,13.575000,Tops & Tees,Burnside Men's Operate Burnside's Plaid Woven ...,Burnside,25.000000,Men,A44E2CF4330786124DEFE8C615147077,9
1272,3915,25.315781,Dresses,FUNFASH NEW SLIMMING PINK BLACK LONG MAXI COCK...,Funfash,59.990002,Women,39539F630A3B94D3ED61EA9D04C9BB05,1
10031,10823,10.240000,Intimates,Calvin Klein Women's Sexy Signature Bikini,Calvin Klein,20.000000,Women,1F47CEF5E38C952F94C5D61726027439,3
15723,5133,14.425190,Pants & Capris,Woman Within Plus Size Tall pants in stretchy ...,Woman Within,29.990000,Women,99064BA6631E279D4A74622DF99657D6,5
14464,14523,25.207020,Maternity,Anita Maternity Women's Softcup Nursing Bra #...,Anita,64.139999,Women,44F455185E5AE730F5E12534AAAA5E02,5


In [91]:
"""
La variabile cost è il costo sostenuto dall'azienda per acquistare o produrre quel prodotto.
È un valore continuo, non arrotondato, e spesso deriva da:
- contratti con fornitori
- costi medi ponderati
- costi di produzione
- costi di importazione
- conversioni di valuta
- calcoli interni dell'azienda

Per questo ha molti decimali. Non è un errore, è semplicemente un valore “raw”, non arrotondato.
"""

"\nLa variabile cost è il costo sostenuto dall'azienda per acquistare o produrre quel prodotto.\nÈ un valore continuo, non arrotondato, e spesso deriva da:\n- contratti con fornitori\n- costi medi ponderati\n- costi di produzione\n- costi di importazione\n- conversioni di valuta\n- calcoli interni dell'azienda\n\nPer questo ha molti decimali. Non è un errore, è semplicemente un valore “raw”, non arrotondato.\n"

In [92]:
# Arrotondiamo cost e retail_price
prod["cost"]=prod["cost"].round(2)
prod["retail_price"]=prod["retail_price"].round(2)

In [93]:
# Togliamo sku
prod=prod[["id", "cost", "category", "name", "brand", "retail_price", "department", "distribution_center_id"]]

In [94]:
prod.sample(5)

,id,cost,category,name,brand,retail_price,department,distribution_center_id
12392,13118,16.84,Swim,Speedo Women's Active Tank Dress Coverup,Speedo,38.99,Women,4
6559,18989,28.78,Sweaters,Alternative Men's Hal Cardigan,Alternative,56.00,Men,2
25684,7856,4.35,Blazers & Jackets,Allegra K Women Deep V Neck Long Sleeves Butto...,Allegra K,10.41,Women,9
956,26225,23.92,Underwear,TRUNKS Men's Swami Short,TRUNKS,54.00,Men,1
10885,2449,20.88,Active,Moving Comfort Women's Maia Bra,Moving Comfort,46.00,Women,3


In [95]:
prod.to_csv("..\\data\\clean_data\\products_clean.csv", index=False)

### Users

In [96]:
users.info()
# city ha Dati Mancanti

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   id              100000 non-null  int64
 1   first_name      100000 non-null  str  
 2   last_name       100000 non-null  str  
 3   age             100000 non-null  int64
 4   gender          100000 non-null  str  
 5   state           100000 non-null  str  
 6   city            99005 non-null   str  
 7   country         100000 non-null  str  
 8   traffic_source  100000 non-null  str  
dtypes: int64(2), str(7)
memory usage: 6.9 MB


In [97]:
users["city"]=users["city"].fillna("Unknown")

In [103]:
users["age"].describe()

count    100000.000000
mean         40.949600
std          17.023095
min          12.000000
25%          26.000000
50%          41.000000
75%          56.000000
max          70.000000
Name: age, dtype: float64

In [ ]:
# Aggiunta categorie età per future analisi
bins = [0, 20, 30, 40, 50, 60, 70, 80, np.inf] 
labels = [ "Under 20", "Twenties", "Thirties", "Forties", "Fifties", "Sixties", "Seventies", "Over 80" ] 
users["age_group"] = pd.cut(users["age"], bins=bins, labels=labels, right=False)

In [105]:
users.sample(5)

,id,first_name,last_name,age,gender,state,city,country,traffic_source,age_group
33184,93750,Tiffany,Green,60,F,Guangdong,Changsha,China,Search,Sixties
83270,22420,Andrea,Lewis,28,F,Shanxi,Chengdu,China,Search,Twenties
84228,60594,Donald,Williams,63,M,Sichuan,Shenzhen,China,Display,Sixties
72558,95919,Jordan,Vargas,50,F,Provence-Alpes-Côte d'Azur,Berre-l'Étang,France,Search,Fifties
60739,72400,Lori,Gardner,12,F,Nevada,Las Vegas,United States,Search,Under 20


In [106]:
users.to_csv("..\\data\\clean_data\\users_clean.csv", index=False)